# F01 偏航次数增加 细化

In [4]:
import numpy as np
import pandas as pd
import yaml
import os
from constans import TURB_NUM, TURB_ATTRIBUTES_OLD
from F01_yaw_up import read_turb_data, yaw_num_counter_torlence

ImportError: cannot import name 'TURB_ATTRIBUTES' from 'constans' (d:\0gzml\VScode\turb_wake_improve\constans.py)

In [2]:
with open('config.yaml', 'r') as file:
    config = yaml.safe_load(file)
all_entries = os.listdir(config['data']['splited_data_new'])

1. 每天偏航角统计

In [10]:
def read_day_data(file_path, file_name):
    full_path = os.path.join(file_path, file_name)
    data = pd.read_csv(full_path, parse_dates=['timestamp'])
    data.set_index('timestamp', inplace=True)
    # turb_data = data.iloc[:, [range(turb_num*TURB_ATTRIBUTES, (turb_num + 1)*TURB_ATTRIBUTES)]] # 选取该机组数据
    resampler = data.resample('D')
    day_data_groups = {date: group for date, group in resampler if not group.empty}

    return day_data_groups

results = []
for file_name in all_entries:
    all_days_dict = read_day_data(config['data']['splited_data_new'], file_name)
    for day_timestamp, day_df in all_days_dict.items():
        file_dict = {'day': day_timestamp}
        for turb_num in range(1, TURB_NUM + 1):
            yaw_wd_diff =  yaw_num_counter_torlence(day_df, turb_num)
            file_dict[f'Turb_{turb_num}'] = yaw_wd_diff
        results.append(file_dict)
df_yaw_wd = pd.DataFrame(results)
df_yaw_wd.to_csv(f'./F01_yaw/yaw_day/yaw_day', index = False)


2. 机组偏航一致性

In [16]:
def get_yaw_consistency(data, turb_num1, turb_num2):
    yaw1 = data[f'{turb_num1}_position']
    yaw2 = data[f'{turb_num2}_position']
    status1 = data[f'{turb_num1}_turbine_status']
    status2 = data[f'{turb_num2}_turbine_status']
    wd_diff = np.abs( yaw1 - yaw2) % 360
    shortest_diff = np.where(wd_diff > 180, 360 - wd_diff, wd_diff)
    shortest_diff[(status1 != 38) | (status2 != 38)] = np.nan
    return (shortest_diff)

results = []
for file_name in all_entries:
    file_dict = {'file_name': file_name}
    data = read_turb_data(config['data']['splited_data_new'], file_name)
    for turb_num in range(1, TURB_NUM):
        # yaw_wd_diff =  get_yaw_consistency(data, turb_num, turb_num + 1)
        yaw_wd_diff =  np.nanstd(get_yaw_consistency(data, turb_num, turb_num + 1))
        file_dict[f'F_{turb_num},F_{turb_num + 1}'] = yaw_wd_diff
    # df_yaw_wd = pd.DataFrame(file_dict)
    # df_yaw_wd.to_csv(f'./F01_yaw/yaw_diff/yaw_consistency_{file_name}', index = False)
    results.append(file_dict)
df_yaw_wd = pd.DataFrame(results)
df_yaw_wd.to_csv('./F01_yaw/yaw_diff/yaw_consistency.csv', index = False)